# Encoding categorical features

Scroll down for exercise.

In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

Categorical features are either:

1. **Ordinal**: order is implied
   - Examples: Education Level, Customer Rating, Income Level
   - Use [`OrdinalEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html)
2. **Nominal**: no order is implied
   - Examples: Gender, Marital Status, Color, Brand, Favorite Sport
   - Use [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

## OrdinalEncoder

Ordinal encoding can give the model information about the ranking of the values of this feature such as shirt size: `xs < s < m < L`.

In [2]:
df = pd.DataFrame({
    'risk': ['low', 'medium', 'low', 'low', 'high'],
    'class': ['1st', '3rd', '2nd', '1st', '3rd'],
})

In [3]:
from sklearn.preprocessing import OrdinalEncoder

# Specify the order of categories
categories = [
    ['low', 'medium', 'high'], # <-- categories of first feature
    ['1st', '2nd', '3rd'],     # <-- categories of second feature
] # <-- this is just a list of lists of strings (i.e., list[list[str]])

encoder = OrdinalEncoder(
    categories=categories, # <-- specify the categories
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
# We want a pandas DataFrame as output rather than a NumPy array (default)
encoder = encoder.set_output(transform='pandas')

In [4]:
encoder.fit_transform(df)

,risk,class
0,0.0,0.0
1,1.0,2.0
2,0.0,1.0
3,0.0,0.0
4,2.0,2.0


## OneHotEncoder

The categorical feature is expanded into multiple binary features, one for each value, where:

- `1` denotes the **existence** of the value
- `0` denotes the **absence** of the value

Example: for the feature color of 3 possible values: `Red`, `Green`, `Blue`.

In [5]:
df_train = pd.DataFrame({
    'color': ['Red', 'Blue', 'Green', 'Green']
})

In [6]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder instance
encoder = OneHotEncoder(
    handle_unknown='infrequent_if_exist',
    sparse_output=False,    # <-- output is a dense array
)
encoder.set_output(transform='pandas')

# Fit and transform the data
encoder.fit(df_train)

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'infrequent_if_exist'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",No

In [7]:
df_test = pd.DataFrame({
    'color': ['Blue', 'Green', 'dragonfruit']
})

# 4. Transform
result = encoder.transform(df_test)
result

,color_Blue,color_Green,color_Red
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,0.0


## Problems to handle in One-Hot Encoding (OHE)

**Cardinality**: The number of unique values in a feature.

Consider a dataset with:

- $m$ samples 
- $C$ unique categories (sum of all unique categories across all categorical nominal features)

**Error: OOM (Out-of-memory)**: Each sample is represented by a vector of length $C$. For high cardinality (say, $C > 1000$), this results in a massive, mostly empty $m \times C$ matrix, which is memory intensive.

As an example, we will calculate the number of columns and memory added by encoding the categorical features on the [**Ames Housing dataset**](https://www.openml.org/search?type=data&status=active&id=41211&sort=runs):

| **Metric** | **Before Encoding** | **After Encoding (OHE)** | **Increase** |
| --- | --- | --- | --- |
| **Column Count** | ~46 | ~318 | **~6.9x** |
| **Memory Usage** | ~0.15 MB | ~7.10 MB | **~47.3x** |

For deatils, see the [OHE Problem Demo](ohe_problem_demo.ipynb).

### Solution

- Use `min_frequency` to cut down on the number of columns by grouping infrequent categories into one "infrequent" column.
- Use `max_categories` to limit the number of columns (including the "infrequent" column).

`OneHotEncoder` (and `OrdinalEncoder`) support aggregating infrequent categories into a single output for each feature. As shown in the table below:

| **Parameter**        | **Type** | **Rule**     | **Description**                                                                              |
| -------------------- | -------- | ------------ | -------------------------------------------------------------------------------------------- |
| **`min_frequency`**  | `int`    | $\ge 1$      | Categories with a count lower than this integer are considered infrequent.                   |
| _                    | `float`  | $(0.0, 1.0)$ | Categories with a count lower than this fraction of total samples are considered infrequent. |
| **`max_categories`** | `int`    | $> 1$        | Limits the total number of output features, including the "infrequent" category.             |
| _                    | `None`   | Default      | No upper limit is placed on the number of output features.                                   |

In [8]:
# Note: multiplying a list by an integer repeats the list that many times
X = np.array([['cat'] * 20 + ['rabbit'] * 10 + ['snake'] * 6 + ['dragon'] * 3 + ['dinosaur'] * 2], dtype=str).T
X.shape

(41, 1)

In [9]:
enc = preprocessing.OneHotEncoder(
    min_frequency=6,
    max_categories=3,
    handle_unknown='infrequent_if_exist',
    sparse_output=False,
).fit(X)

# enc.set_output(transform='pandas')

print("Categories:", enc.categories_)
print("Infrequent categories:", enc.infrequent_categories_)

Categories: [array(['cat', 'dinosaur', 'dragon', 'rabbit', 'snake'], dtype='<U8')]
Infrequent categories: [array(['dinosaur', 'dragon', 'snake'], dtype='<U8')]


- Notice how `'dragon'` and `'dinosaur'` would both map to: `[0., 0., 1.]` (the shared encoding of all infrequent categories).
- Since we restricted our `max_categories` parameter to 3, even though `'snake'` satisfies the `min_frequency` parameter, it is not included in the output. Because, the third category is being used for the infrequent categories.

In [10]:
enc.transform(np.array([
    ['rabbit'],
    ['rabbit'],
    ['cat'],
    ['snake'],
    ['dragon'],
    ['dinosaur'],
]))

array([[0., 1., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])

## Extra: Geo-Encoding

For location-based data, **Geocoding**, which outputs latitude and longitude information is often a better choice than any other method.

---

To learn more, see the User Guide on scikit-learn docs: [7.3.4. Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features).

# Exercise: Categorical feature encoding

Dataset: [Ames Housing dataset](https://www.openml.org/search?type=data&status=active&id=42165)

1. Download the dataset and load it into a pandas DataFrame.
2. Encode the categorical features using the `OrdinalEncoder` and `OneHotEncoder` using the `fit` and `transform` methods

Consider the following columns:

- `SalePrice`: The property's sale price in dollars. This is the **target variable**.
- `LotArea`: Lot size in square feet
- `MSZoning`: Identifies the general zoning classification of the sale.
  - Categories:
    - `A`	    Agriculture
    - `C`	    Commercial
    - `FV`	Floating Village Residential
    - `I`	    Industrial
    - `RH`	Residential High Density
    - `RL`	Residential Low Density
    - `RP`	Residential Low Density Park 
    - `RM`	Residential Medium Density
- `LandSlope`: Slope of property
  - Categories:
    - `Gtl`	Gentle slope
    - `Mod`	Moderate Slope	
    - `Sev`	Severe Slope

**Optional**: Split the data into training and testing sets using the [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function, and use the `fit` on the training set and `transform` on the testing set.

In [11]:
from sklearn.datasets import fetch_openml

ames_housing = fetch_openml(name="house_prices", as_frame=True)
data = ames_housing.data
target = ames_housing.target

data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [12]:
# INSERT CODE HERE